In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [19]:
import peft
import transformers

print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

PEFT: 0.19.1
Transformers: 5.12.1


In [ ]:
# !pip install -q transformers peft accelerate bitsandbytes sentencepiece

In [20]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import PeftModel

from IPython.display import Markdown, display

# **Inference Testing**

In [5]:
# Configurations
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_MODEL = "igmoiiz/math-tutor-qwen-lora"

In [6]:
# Checking For GPU
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [7]:
# Loading the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [8]:
# Loading the BASE MODEL
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [9]:
# Loading the Fine Tuned Adapter from igmoiiz Hugging Face
model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

model.eval()

print("Math Tutor Loaded Successfully!")

adapter_model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

Math Tutor Loaded Successfully!


In [10]:
# Counting Parameters
trainable = 0
total = 0

for _, param in model.named_parameters():
    total += param.numel()
    if param.requires_grad:
        trainable += param.numel()

print(f"Total Parameters: {total:,}")
print(f"Trainable Parameters: {trainable:,}")

Total Parameters: 1,548,072,448
Trainable Parameters: 0


In [11]:
# Inference Function
def ask(question, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful math tutor. Solve the problem step by step."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [21]:
# First Question Test
question = """
A train travels 80 km in 2 hours.
What is its average speed?
"""

response = ask(question)

print(display(Markdown(response)))

To find the average speed of the train, we use the formula for average speed:

\[
\text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}}
\]

Given:
- Total distance traveled: \(80\) km
- Total time taken: \(2\) hours

Now, plug these values into the formula:

\[
\text{Average Speed} = \frac{80 \text{ km}}{2 \text{ hours}}
\]

Perform the division:

\[
\text{Average Speed} = 40 \text{ km/h}
\]

So, the average speed of the train is \(40\) kilometers per hour (km/h).

None


In [13]:
# Hard Question Test
print(
    ask(
        """
Sarah bought 8 notebooks.

Each notebook costs $4.

She also bought 3 pens costing $2 each.

How much money did she spend altogether?
"""
    )
)

To find out how much money Sarah spent in total, we need to calculate the cost of the notebooks and the pens separately, then add them together.

First, let's calculate the cost of the notebooks:
- Sarah bought 8 notebooks.
- Each notebook costs $4.

So, the total cost for the notebooks is:
\[ 8 \text{ notebooks} \times \$4 \text{ per notebook} = \$32 \]

Next, let's calculate the cost of the pens:
- Sarah bought 3 pens.
- Each pen costs $2.

So, the total cost for the pens is:
\[ 3 \text{ pens} \times \$2 \text{ per pen} = \$6 \]

Now, we add the cost of the notebooks and the pens to get the total amount spent:
\[ \$32 + \$6 = \$38 \]

Therefore, Sarah spent a total of $38.


In [14]:
# Multi Step Reasoning
print(
    ask(
        """
Ali had 50 marbles.

He gave 18 to his friend.

Then he bought 27 more.

How many marbles does he have now?
"""
    )
)

Let's solve this step by step:

1) Ali starts with 50 marbles.
   This is our initial number of marbles: 50

2) He gives away 18 marbles:
   50 - 18 = 32 marbles remaining

3) Then he buys 27 more marbles:
   32 + 27 = 59 marbles in total

So, Ali has 59 marbles now.


In [15]:
# Fraction Problem
print(
    ask(
        """
A pizza is cut into 8 equal slices.

Ahmed eats 3 slices.

What fraction of the pizza remains?
"""
    )
)

To determine what fraction of the pizza remains after Ahmed eats some slices, we can follow these steps:

1. **Identify the total number of slices**: The pizza is cut into 8 equal slices.
2. **Determine how many slices were eaten**: Ahmed ate 3 slices.
3. **Calculate the number of slices remaining**: Subtract the number of slices eaten from the total number of slices:
   \[
   8 - 3 = 5
   \]
   So, there are 5 slices remaining.
4. **Express the remaining slices as a fraction of the whole pizza**: Since the whole pizza consists of 8 slices, the fraction of the pizza that remains is:
   \[
   \frac{5}{8}
   \]

Therefore, the fraction of the pizza that remains is \(\boxed{\frac{5}{8}}\).


In [16]:
# Geometry Problem
print(
    ask(
        """
A rectangle has length 15 cm and width 8 cm.

Find its area.
"""
    )
)

To find the area of a rectangle, you use the formula:

\[ \text{Area} = \text{length} \times \text{width} \]

Given:
- Length (\( l \)) = 15 cm
- Width (\( w \)) = 8 cm

Now plug these values into the formula:

\[ \text{Area} = 15 \, \text{cm} \times 8 \, \text{cm} \]

\[ \text{Area} = 120 \, \text{cm}^2 \]

So, the area of the rectangle is \( 120 \, \text{square centimeters} \).
